# Draft-02: Titanic CatBoost + Feature Engineering (Kaggle)

Score-improvement draft built on the same Kaggle workflow as `01`, but with stronger features, 5-fold CV, threshold tuning, and a CatBoost model that is usually a better fit for Titanic than plain logistic regression.


In [ ]:
import importlib.util
import subprocess
import sys


def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is not None:
        print(f"{pkg} is already installed")
        return
    run_cmd([sys.executable, "-m", "pip", "install", pkg])


ensure_package("dotenv", "python-dotenv")
ensure_package("kaggle")
ensure_package("catboost")


In [ ]:
import os
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


In [ ]:
COMPETITION = "titanic"
RUN_DOWNLOAD = True
RUN_SUBMISSION = True
FORCE_DOWNLOAD = True
RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLD_GRID = np.linspace(0.35, 0.65, 61)

ID_COL = "PassengerId"
LABEL_COL = "Survived"
NOTEBOOK_SLUG = "titnic-02-catboost-features"

BASE_PARAMS = {
    "loss_function": "Logloss",
    "eval_metric": "Accuracy",
    "iterations": 1200,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 4.0,
    "random_strength": 0.5,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.85,
    "verbose": False,
    "allow_writing_files": False,
}


def load_env_from_candidates():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for directory in candidates:
        env_path = directory / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            print("Loaded env from", env_path)
            return env_path
    print("No .env file found in notebook parents, using current environment")
    return None


def resolve_data_paths_fallback():
    search_roots = [
        Path("/kaggle/input/titanic"),
        Path("/kaggle/input"),
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd().parent,
        Path.cwd().parent / "data",
    ]
    candidate_triplets = []
    for base in search_roots:
        candidate_triplets.extend(
            [
                (base / "train.csv", base / "test.csv", base / "gender_submission.csv"),
                (
                    base / "titanic" / "train.csv",
                    base / "titanic" / "test.csv",
                    base / "titanic" / "gender_submission.csv",
                ),
                (
                    base / "data" / "train.csv",
                    base / "data" / "test.csv",
                    base / "data" / "gender_submission.csv",
                ),
            ]
        )
    for train_path, test_path, sample_path in candidate_triplets:
        if train_path.exists() and test_path.exists() and sample_path.exists():
            return train_path, test_path, sample_path
    raise FileNotFoundError(
        "Could not find Titanic train/test/gender_submission files in Kaggle input or local folders"
    )


def prepare_competition_data(api_client, competition, data_dir, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f"{competition}.zip"
    extract_dir = data_dir / competition

    if force_download or not zip_path.exists():
        files_resp = api_client.competition_list_files(competition)
        try:
            api_client.competition_download_files(
                competition=competition,
                path=str(data_dir),
                force=force_download,
                quiet=False,
            )
            print("Competition zip download complete")
        except Exception as download_error:
            print(
                "Bulk download failed, fallback to per-file download:", download_error
            )
            for file_obj in files_resp.files:
                api_client.competition_download_file(
                    competition=competition,
                    file_name=file_obj.name,
                    path=str(data_dir),
                    force=force_download,
                    quiet=False,
                )

    if zip_path.exists() and (
        not extract_dir.exists() or not any(extract_dir.iterdir())
    ):
        extract_dir.mkdir(parents=True, exist_ok=True)
        with ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

    train_path = extract_dir / "train.csv"
    test_path = extract_dir / "test.csv"
    sample_path = extract_dir / "gender_submission.csv"

    if train_path.exists() and test_path.exists() and sample_path.exists():
        return train_path, test_path, sample_path, zip_path, extract_dir

    fallback_train = data_dir / "train.csv"
    fallback_test = data_dir / "test.csv"
    fallback_sample = data_dir / "gender_submission.csv"
    if fallback_train.exists() and fallback_test.exists() and fallback_sample.exists():
        return fallback_train, fallback_test, fallback_sample, zip_path, extract_dir

    raise FileNotFoundError(
        "Could not resolve Titanic train/test/gender_submission files after download"
    )


In [ ]:
load_env_from_candidates()
os.environ.pop("KAGGLE_API_TOKEN", None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print("Authenticated user:", api.get_config_value("username"))
except Exception as auth_error:
    print("Kaggle API auth not ready:", auth_error)
    print("Falling back to existing /kaggle/input or local data if available")

existing_paths = None
try:
    existing_paths = resolve_data_paths_fallback()
except FileNotFoundError:
    existing_paths = None

run_download_effective = RUN_DOWNLOAD
if RUN_DOWNLOAD is None:
    run_download_effective = existing_paths is None

if run_download_effective:
    if api is None:
        raise RuntimeError(
            "Auto download is needed but Kaggle API auth is not available"
        )
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / "data"
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH, ZIP_PATH, EXTRACT_DIR = (
        prepare_competition_data(
            api_client=api,
            competition=COMPETITION,
            data_dir=DATA_DIR,
            force_download=FORCE_DOWNLOAD,
        )
    )
else:
    if existing_paths is None:
        raise FileNotFoundError(
            "RUN_DOWNLOAD resolved to False, but Titanic files were not found"
        )
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = existing_paths
    BASE_DIR = TRAIN_PATH.parent
    ZIP_PATH = None
    EXTRACT_DIR = TRAIN_PATH.parent

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_PATH = OUTPUT_DIR / f"submission_{NOTEBOOK_SLUG}.csv"
DIAGNOSTICS_PATH = OUTPUT_DIR / f"diagnostics_{NOTEBOOK_SLUG}.csv"
FOLD_TABLE_PATH = OUTPUT_DIR / f"fold_scores_{NOTEBOOK_SLUG}.csv"
OOF_PATH = OUTPUT_DIR / f"oof_predictions_{NOTEBOOK_SLUG}.csv"
THRESHOLD_PATH = OUTPUT_DIR / f"threshold_grid_{NOTEBOOK_SLUG}.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / f"feature_importance_{NOTEBOOK_SLUG}.csv"

config_df = pd.DataFrame(
    [
        {
            "competition": COMPETITION,
            "run_download_effective": run_download_effective,
            "run_submission": RUN_SUBMISSION,
            "force_download": FORCE_DOWNLOAD,
            "random_state": RANDOM_STATE,
            "n_splits": N_SPLITS,
            "train_path": str(TRAIN_PATH),
            "test_path": str(TEST_PATH),
            "sample_path": str(SAMPLE_PATH),
            "output_file": str(OUTPUT_PATH),
        }
    ]
)
display(config_df)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)
display(train_df.head())
display(sample_submission.head())

missing_df = train_df.isna().sum().rename("missing").to_frame()
missing_df["missing_pct"] = missing_df["missing"] / len(train_df)
display(missing_df[missing_df["missing"] > 0].sort_values("missing", ascending=False))


In [ ]:
def engineer_features(train_raw, test_raw):
    train_part = train_raw.copy()
    test_part = test_raw.copy()
    train_part["_dataset"] = "train"
    test_part["_dataset"] = "test"
    combined = pd.concat([train_part, test_part], axis=0, ignore_index=True, sort=False)

    combined["Title"] = (
        combined["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False).fillna("Unknown")
    )
    common_titles = {"Mr", "Mrs", "Miss", "Master"}
    combined["Title"] = combined["Title"].where(
        combined["Title"].isin(common_titles), "Rare"
    )

    combined["Surname"] = combined["Name"].str.split(",").str[0].fillna("Unknown")
    combined["FamilySize"] = combined["SibSp"] + combined["Parch"] + 1
    combined["IsAlone"] = (combined["FamilySize"] == 1).astype(int)
    combined["FamilyCategory"] = np.where(
        combined["FamilySize"] == 1,
        "solo",
        np.where(combined["FamilySize"] <= 4, "small", "large"),
    )
    combined["CabinDeck"] = combined["Cabin"].fillna("U").astype(str).str[0]
    combined["CabinKnown"] = combined["Cabin"].notna().astype(int)
    combined["NameLength"] = combined["Name"].fillna("").str.len()

    ticket_clean = combined["Ticket"].fillna("NONE").astype(str).str.upper()
    ticket_clean = (
        ticket_clean.str.replace(r"[\./]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    combined["TicketPrefix"] = (
        ticket_clean.str.replace(r"\d+", "", regex=True).str.strip().replace("", "NUM")
    )
    ticket_counts = combined["Ticket"].fillna("NONE").value_counts()
    combined["TicketGroupSize"] = (
        combined["Ticket"].fillna("NONE").map(ticket_counts).astype(int)
    )

    embarked_mode = combined["Embarked"].dropna().mode().iloc[0]
    combined["Embarked"] = combined["Embarked"].fillna(embarked_mode)
    combined["Fare"] = combined["Fare"].fillna(
        combined.groupby("Pclass")["Fare"].transform("median")
    )
    combined["Fare"] = combined["Fare"].fillna(combined["Fare"].median())
    combined["Age"] = combined["Age"].fillna(
        combined.groupby(["Title", "Pclass"])["Age"].transform("median")
    )
    combined["Age"] = combined["Age"].fillna(
        combined.groupby("Pclass")["Age"].transform("median")
    )
    combined["Age"] = combined["Age"].fillna(combined["Age"].median())

    combined["FarePerPerson"] = combined["Fare"] / combined["FamilySize"].clip(lower=1)
    combined["AgeClass"] = combined["Age"] * combined["Pclass"]
    combined["AgeBand"] = pd.cut(
        combined["Age"],
        bins=[-1, 5, 12, 18, 30, 45, 60, 100],
        labels=["baby", "child", "teen", "young_adult", "adult", "mid_age", "senior"],
    ).astype(str)
    combined["FareBand"] = pd.qcut(
        combined["Fare"].rank(method="first"),
        q=4,
        labels=["fare_q1", "fare_q2", "fare_q3", "fare_q4"],
    ).astype(str)

    feature_cols = [
        "Pclass",
        "Sex",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "Embarked",
        "Title",
        "FamilySize",
        "IsAlone",
        "FamilyCategory",
        "CabinDeck",
        "CabinKnown",
        "NameLength",
        "TicketPrefix",
        "TicketGroupSize",
        "FarePerPerson",
        "AgeClass",
        "AgeBand",
        "FareBand",
    ]
    categorical_features = [
        "Sex",
        "Embarked",
        "Title",
        "FamilyCategory",
        "CabinDeck",
        "TicketPrefix",
        "AgeBand",
        "FareBand",
    ]

    for col in categorical_features:
        combined[col] = combined[col].fillna("Missing").astype(str)

    train_features = (
        combined[combined["_dataset"] == "train"].copy().reset_index(drop=True)
    )
    test_features = (
        combined[combined["_dataset"] == "test"].copy().reset_index(drop=True)
    )
    return train_features, test_features, feature_cols, categorical_features


train_features, test_features, feature_cols, categorical_features = engineer_features(
    train_df, test_df
)
X = train_features[feature_cols].copy()
y = train_features[LABEL_COL].copy()
X_test = test_features[feature_cols].copy()

print("Feature count:", len(feature_cols))
display(train_features[feature_cols + [LABEL_COL]].head())


In [ ]:
survival_by_title = (
    train_features.groupby("Title")[LABEL_COL].mean().sort_values(ascending=False)
)
survival_by_family = (
    train_features.groupby("FamilyCategory")[LABEL_COL]
    .mean()
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(survival_by_title.index, survival_by_title.values)
axes[0].set_title("Survival Rate by Title")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_ylim(0, 1)

axes[1].bar(survival_by_family.index, survival_by_family.values)
axes[1].set_title("Survival Rate by Family Category")
axes[1].set_ylim(0, 1)

train_features.groupby(LABEL_COL)["Fare"].mean().plot(kind="bar", ax=axes[2])
axes[2].set_title("Average Fare by Survival")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_pred_proba = np.zeros(len(X), dtype=float)
test_fold_pred_probas = []
fold_rows = []
importance_parts = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_train = X.iloc[train_idx].copy()
    y_train = y.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx].copy()

    model = CatBoostClassifier(**BASE_PARAMS, random_seed=RANDOM_STATE + fold)
    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features,
        eval_set=(X_val, y_val),
        use_best_model=True,
        early_stopping_rounds=100,
        verbose=False,
    )

    val_pred_proba = model.predict_proba(X_val)[:, 1]
    test_pred_proba = model.predict_proba(X_test)[:, 1]
    oof_pred_proba[val_idx] = val_pred_proba
    test_fold_pred_probas.append(test_pred_proba)

    val_pred = (val_pred_proba >= 0.5).astype(int)
    fold_accuracy = accuracy_score(y_val, val_pred)
    fold_f1 = f1_score(y_val, val_pred)
    best_iteration = model.get_best_iteration()
    if best_iteration is None or best_iteration <= 0:
        best_iteration = BASE_PARAMS["iterations"]

    fold_rows.append(
        {
            "fold": fold,
            "accuracy_at_0_50": fold_accuracy,
            "f1_at_0_50": fold_f1,
            "best_iteration": int(best_iteration),
            "train_rows": int(len(train_idx)),
            "validation_rows": int(len(val_idx)),
        }
    )

    importance_parts.append(
        pd.DataFrame(
            {
                "fold": fold,
                "feature": feature_cols,
                "importance": model.get_feature_importance(),
            }
        )
    )

fold_df = pd.DataFrame(fold_rows)
display(fold_df)


In [ ]:
threshold_rows = []
for threshold in THRESHOLD_GRID:
    oof_pred = (oof_pred_proba >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            "accuracy": accuracy_score(y, oof_pred),
            "f1": f1_score(y, oof_pred),
            "positive_rate": float(oof_pred.mean()),
        }
    )

threshold_df = (
    pd.DataFrame(threshold_rows)
    .sort_values(["accuracy", "f1", "threshold"], ascending=[False, False, True])
    .reset_index(drop=True)
)
best_threshold = float(threshold_df.loc[0, "threshold"])
oof_pred = (oof_pred_proba >= best_threshold).astype(int)
oof_accuracy = accuracy_score(y, oof_pred)
oof_f1 = f1_score(y, oof_pred)

print(f"Best threshold: {best_threshold:.3f}")
print(f"OOF accuracy: {oof_accuracy:.4f}")
print(f"OOF f1: {oof_f1:.4f}")
print("Classification report:")
print(classification_report(y, oof_pred))

display(threshold_df.head(10))


In [ ]:
cm = confusion_matrix(y, oof_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("OOF Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

oof_df = pd.DataFrame(
    {
        ID_COL: train_df[ID_COL],
        "oof_pred_proba": oof_pred_proba,
        "oof_pred": oof_pred,
        "actual": y,
    }
)
oof_df.to_csv(OOF_PATH, index=False)
fold_df.to_csv(FOLD_TABLE_PATH, index=False)
threshold_df.to_csv(THRESHOLD_PATH, index=False)

importance_df = pd.concat(importance_parts, ignore_index=True)
importance_summary = (
    importance_df.groupby("feature")["importance"]
    .agg(["mean", "std"])
    .reset_index()
)
importance_summary.columns = ["feature", "importance_mean", "importance_std"]
importance_summary = importance_summary.sort_values("importance_mean", ascending=False)
importance_summary.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

diagnostics_df = pd.DataFrame(
    [
        {
            "oof_accuracy": oof_accuracy,
            "oof_f1": oof_f1,
            "best_threshold": best_threshold,
            "mean_fold_accuracy_at_0_50": fold_df["accuracy_at_0_50"].mean(),
            "std_fold_accuracy_at_0_50": fold_df["accuracy_at_0_50"].std(),
            "mean_best_iteration": fold_df["best_iteration"].mean(),
        }
    ]
)
diagnostics_df.to_csv(DIAGNOSTICS_PATH, index=False)

display(diagnostics_df)
display(importance_summary.head(15))

print("Diagnostics saved to", DIAGNOSTICS_PATH)
print("Fold scores saved to", FOLD_TABLE_PATH)
print("OOF predictions saved to", OOF_PATH)
print("Threshold grid saved to", THRESHOLD_PATH)
print("Feature importance saved to", FEATURE_IMPORTANCE_PATH)


In [ ]:
final_iterations = int(max(300, round(fold_df["best_iteration"].median())))
final_model = CatBoostClassifier(
    **BASE_PARAMS, iterations=final_iterations, random_seed=RANDOM_STATE
)
final_model.fit(X, y, cat_features=categorical_features, verbose=False)

test_pred_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_pred_proba >= best_threshold).astype(int)

submission_df = sample_submission.copy()
submission_df[LABEL_COL] = test_pred
submission_df.to_csv(OUTPUT_PATH, index=False)

display(submission_df.head(10))
print("Submission file saved to", OUTPUT_PATH)
print("Predicted survival rate:", f"{submission_df[LABEL_COL].mean():.2%}")
print("Final iterations:", final_iterations)


In [ ]:
if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError("RUN_SUBMISSION=True but Kaggle API auth is not available")
    submit_message = (
        f"{NOTEBOOK_SLUG} catboost features | "
        f"oof_acc={oof_accuracy:.6f} "
        f"oof_f1={oof_f1:.6f} "
        f"thr={best_threshold:.3f} "
        f"iter={final_iterations} "
        f"time={datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S")}Z"
    )
    response = api.competition_submit(
        file_name=str(OUTPUT_PATH),
        message=submit_message,
        competition=COMPETITION,
        quiet=False,
    )
    print("Submission response:", response)
else:
    print("RUN_SUBMISSION is False. File is ready at", OUTPUT_PATH)


## Notes

- `RUN_DOWNLOAD = None` auto-detects whether Titanic files already exist and downloads only when needed.
- `RUN_SUBMISSION = False` by default so you can inspect outputs before burning a Kaggle submission.
- This draft uses 5-fold OOF accuracy for a less misleading validation estimate than the single split in `01`.
